<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2007%20-%202026-04-29%20-%20Pandas%20II/clase_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 7 · Pandas II

Si ayer leyeron datos, hoy los van a reorganizar. `groupby()` es 80% del trabajo real — no es una exageración. Hoy también limpian datos: nulos, duplicados, tipos. Es el trabajo feo que nadie cuenta pero todos hacen.

> **Hoy haces** · Los tres ejercicios sobre groupby, limpieza y columnas derivadas, las dos pausas y el checkpoint. ~2h30.
>
> **Entrega** · El notebook ejecutado al repo del equipo antes de la próxima clase.

In [ ]:
import random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
print("Setup completo")
print(f"pandas {pd.__version__} · numpy {np.__version__}")

## Resumen de supervivencia por sexo

¿Cuántos sobrevivieron en cada grupo? ¿Cuál fue la tasa de supervivencia? Ahora lo hacemos con `groupby()` + `agg()`:

In [ ]:
titanic = sns.load_dataset("titanic")

# Agrupación simple
por_sexo = titanic.groupby("sex")["survived"].agg(["count", "sum", "mean"])
por_sexo.columns = ["Total", "Sobrevivieron", "Tasa"]
por_sexo["Tasa"] = (por_sexo["Tasa"] * 100).round(1)

print(por_sexo)

`groupby()` divide el DataFrame por categorías, luego `agg()` aplica funciones a cada grupo. Es simple pero poderoso — y lo vas a usar en todo.

## Groupby básico

Edad promedio por clase de pasaje. Completa los blancos:

In [ ]:
# Completa los huecos
edad_por_clase = titanic._____("pclass")["age"].____
edad_por_clase = edad_por_clase.round(2)

print("Edad promedio por clase:")
print(edad_por_clase)

In [ ]:
assert len(edad_por_clase) == 3, "Deberías tener 3 clases"
assert edad_por_clase.iloc[0] > 30, "Clase 1 debe tener edad promedio > 30"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# edad_por_clase = titanic.groupby("pclass")["age"].mean()

¿Quiénes eran en promedio más jóvenes en el Titanic? Tercera clase. Más inmigrantes, más jóvenes, menos dinero.

## Agregaciones múltiples

Para cada combinación de sexo + clase, calcula: count, mean, min, max de tarifa. Contexto: análisis de tarifa por grupo:

In [ ]:
# Agrupa por sexo Y pclass, luego aplica agg en fare
fare_stats = titanic.___(["sex", "pclass"])["fare"].___({"count": "count", "mean": "mean", "min": "min", "max": "max"})

print("Tarifa por sexo y clase:")
print(fare_stats.round(2))

In [ ]:
assert fare_stats.shape[0] == 6, f"Deberías tener 6 filas (2 sexos × 3 clases), tienes {fare_stats.shape[0]}"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# fare_stats = titanic.groupby(["sex", "pclass"])["fare"].agg({"count": "count", "mean": "mean", "min": "min", "max": "max"})

¿Cuál es el segmento más caro? ¿El más barato? Mujeres, clase 1: tarifa promedio de USD 106. Hombres, clase 3: USD 26. Eso es 4× de diferencia.

## Limpieza y pipeline

Aquí es donde sucede la magia — sin esta parte, el análisis es garbage. Tienes un dataset con nulos, duplicados y tipos incorrectos. Tu tarea:

1. Identifica nulos por columna y decide cuál remover/imputar.
2. Busca duplicados exactos.
3. Convierte `age` a entero (rellenando nulos con la mediana).
4. Crea una variable derivada: "es_niño" (age < 18).
5. Reporta shape antes y después.

In [ ]:
# Tu código aquí
df = titanic.copy()

print("ANTES:")
print(f"Shape: {df.shape}")
print(f"Nulos por columna:")
print(df.isnull().sum())

# Paso 1: remover columnas con muchos nulos (ej: deck)
# df = df.drop(columns=[...])

# Paso 2: rellenar age con mediana
# edad_mediana = ...
# df["age"] = df["age"].fillna(edad_mediana)

# Paso 3: convertir a int
# df["age"] = df["age"].astype(int)

# Paso 4: crear variable derivada
# df["es_nino"] = df["age"] < 18

print("\nDESPUES:")
print(f"Shape: {df.shape}")
print(f"Nulos: {df.isnull().sum().sum()}")

In [ ]:
# Criterios de aceptación
assert "df" in dir(), "Variable df no existe"
assert "es_nino" in df.columns, "Falta columna es_nino"
assert df["age"].dtype in [int, "int64"], "age debe ser int"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# df = titanic.drop(columns=["deck"]).copy()
# df["age"] = df["age"].fillna(df["age"].median())
# df["age"] = df["age"].astype(int)
# df["es_nino"] = df["age"] < 18

## Antes y después

Visualicemos lo que hicimos:

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
titanic["age"].hist(bins=30, alpha=0.7, color="blue", edgecolor="black")
plt.title("Antes: edades (con nulos)")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")

plt.subplot(1, 2, 2)
df["age"].hist(bins=30, alpha=0.7, color="green", edgecolor="black")
plt.title("Después: edades (limpio)")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")

plt.tight_layout()
plt.show()

## Checkpoint

1. ¿Qué hace `groupby()`? Divide el DataFrame por categorías, luego agrega cada grupo.
2. ¿Cómo rellenas nulos con la mediana? `df["col"].fillna(df["col"].median())`
3. ¿Cómo creas una variable derivada (edad < 18)? `df["es_nino"] = df["age"] < 18`

Si algo no quedó claro, vuelve atrás.

In [ ]:
assert df.shape[0] == titanic.shape[0], "No removiste filas incorrectamente"
print("Checkpoint ✓ — puedes continuar")

## El flujo de limpieza

```
DataFrame crudo
      ↓
  .isnull().sum() — identifica nulos
      ↓
  .fillna() / .drop() — limpia
      ↓
  .groupby() — agrupa
      ↓
  .agg() — resume cada grupo
      ↓
  Respuestas segmentadas
```

## Para recordar

- **`groupby()` + `agg()`** para resumir por categorías.
- **`fillna()`** para imputar nulos (media, mediana, valor constante).
- **`.isnull().sum()`** para diagnosticar datos faltantes.
- **Variables derivadas** con operaciones booleanas: `df["new"] = df["col"] < X`.
- **Pipelines de limpieza:** nulos → duplicados → tipos → derivadas.

## Mañana

Veremos **Estadística descriptiva + visualización** — media, mediana, std, distribuciones, outliers, plots exploratorios. Antes de que cualquier ML tenga sentido, tienes que saber describir lo que tienes.

Para preparar:
- Relee `groupby()` en el HTML si la sintaxis te confundió.
- Practica con `sns.load_dataset("tips")`: agrupa por sexo, calcula mean y std de total_bill.

## Referencias

- [Pandas GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Pandas Data Cleaning](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Data Cleaning Best Practices](https://www.datacamp.com/courses/data-cleaning-in-python)

## Reflexión

En tus propias palabras, explícale a un compañero **por qué la limpieza de datos es el 80% del trabajo en EDA** en máximo 3 oraciones.

> **Recordatorio** · Notebook ejecutado al repo antes de la próxima sesión.

---

## Para tu equipo

**Limpieza y exploración de su dataset:**

- Realicen limpieza básica:
  - Identifiquen y documenten nulos
  - Remuevan duplicados exactos
  - Conviertan tipos si es necesario
  - Creen 2 variables derivadas relevantes para su caso
- Agrupaciones:
  - Busquen **3 groupby interesantes** según su pregunta de negocio
  - Reporten media, count, min, max

**Entregable:** `proyecto_limpieza_exploracion.ipynb` con pipeline completo.